# 🎓 InterviewMate: Fine-Tuning & Web Deployment
This notebook demonstrates the end-to-end workflow to load, fine-tune, and serve our custom AI mock interviewer on a T4 GPU.

In [29]:
import json
import re
import torch

def extract_and_parse_json(text):
    """Robustly extracts and parses a JSON object from text containing potential markdown wrappers."""
    if not text or not isinstance(text, str):
        raise ValueError("Empty or invalid string received from model.")

    # Clean whitespace and potential markdown wrappers
    cleaned = text.strip()

    # Try extracting everything between the first '{' and the last '}'
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)

    return json.loads(cleaned)

# Test inference with the specific database index question and candidate answer
test_role = "Backend Engineer"
test_track = "software_development"
test_experience = "Mid-Level"
test_question = "How do database indexes improve query performance, and what are the trade-offs of using them?"
test_answer = "Database indexes help the database locate rows without scanning the entire table. They can speed up SELECT queries, but require additional storage and can slow down INSERT, UPDATE, and DELETE operations because the indexes must be maintained."

# We MUST structure this prompt exactly like our ChatML training dataset structure!
user_prompt = f"""Candidate Target Role: {test_role}
Experience Level: {test_experience}
Interview Track: {test_track}

Interview Question:
\"{test_question}\"

Candidate's Answer:
\"{test_answer}\"

Scoring Rubric (Evaluate each metric from 1 to 10):
1. relevance (1-10)
2. accuracy_depth (1-10)
3. clarity_structure (1-10)
4. completeness (1-10)

Provide your evaluation strictly as a valid JSON object matching this schema:
{{
  "overall_score": <number 1-10>,
  "scoring_rubric": {{
    "relevance": <1-10>,
    "accuracy_depth": <1-10>,
    "clarity_structure": <1-10>,
    "completeness": <1-10>
  }},
  "strengths": ["<strength 1>", ...],
  "weaknesses": ["<weakness 1>", ...],
  "actionable_feedback": ["<actionable advice 1>", ...],
  "improved_sample_answer": "<ideal answer>",
  "follow_up_question": "<contextual follow-up>"
}}"""

print("🔄 Running test inference against the model with the correct ChatML training format...")
try:
    # Execute model generation using the canonical helper
    raw_output = test_inference(trainer.model, tokenizer, user_prompt)
    print("\n--- Raw Model Output ---")
    print(raw_output)

    # Test our parsing function
    parsed_json = extract_and_parse_json(raw_output)
    print("\n✅ Parsed JSON Feedback Successfully:")
    print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"\n❌ Test failed with error: {str(e)}")

🔄 Running test inference against the model with the correct ChatML training format...

--- Raw Model Output ---
{
  "overall_score": 7,
  "scoring_rubric": {
    "relevance": 8,
    "accuracy_depth": 6,
    "clarity_structure": 7,
    "completeness": 6
  },
  "strengths": [
    "Understands the fundamental concept.",
    "Identifies practical advantages and common tooling."
  ],
  "weaknesses": [
    "Misses quantitative benchmarks or specific architectural patterns.",
    "Fails to address edge-case handling or operational challenges."
  ],
  "actionable_feedback": [
    "Quantify system metrics (throughput, latency).",
    "Discuss failure recovery or operational trade-offs explicitly."
  ],
  "improved_sample_answer": "Indexes store pointers into data blocks so that lookup times are independent of block size. This allows efficient range scans on WHERE clauses. However, they increase write amplification by requiring updates to maintain pointer consistency across all replicas. Index m

In [ ]:
# Test inference with the specific database index question and candidate answer
test_role = "Backend Engineer"
test_track = "software_development"
test_experience = "Mid-Level"
test_question = "How do database indexes improve query performance, and what are the trade-offs of using them?"
test_answer = "Database indexes help the database locate rows without scanning the entire table. They can speed up SELECT queries, but require additional storage and can slow down INSERT, UPDATE, and DELETE operations because the indexes must be maintained."

# Construct the prompt
lines = [
    f"Candidate Target Role: {test_role}",
    f"Experience Level: {test_experience}",
    f"Interview Track: {test_track}",
    "",
    "Interview Question:",
    f"{test_question}",
    "",
    "Candidate's Answer:",
    f"{test_answer}"
]
test_prompt = "\n".join(lines)

print("🔄 Running test inference against the model...")
try:
    # Execute model generation
    raw_output = test_inference(trainer.model, tokenizer, test_prompt)
    print("\n--- Raw Model Output ---")
    print(raw_output)

    # Test our parsing function
    parsed_json = extract_and_parse_json(raw_output)
    print("\n✅ Parsed JSON Feedback Successfully:")
    print(json.dumps(parsed_json, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"\n❌ Test failed with error: {str(e)}")

🔄 Running test inference against the model...

--- Raw Model Output ---
Scoring Rubric (Evaluate each metric from 1 to 10):
1. relevance (1-10)
2. accuracy_depth (1-10)
3. clarity_structure (1-10)
4. completeness (1-10)

Provide your evaluation strictly as a valid JSON object matching this schema:
{
  "overall_score": <number 1-10>,
  "scoring_rubric": {
    "relevance": <1-10>,
    "accuracy_depth": <1-10>,
    "clarity_structure": <1-10>,
    "completeness": <1-10>
  },
  "strengths": ["<strength 1>", ...],
  "weaknesses": ["<weakness 1>", ...],
  "actionable_feedback": ["<actionable advice 1>", ...],
  "improved_sample_answer": "<ideal answer>",
  "follow_up_question": "<contextual follow-up>"
}

❌ Test failed with error: name 'extract_and_parse_json' is not defined


# 🎓 InterviewMate: Fine-Tuning & Cloud Deployment on Google Colab
### AI Mock Interviewer Specialized via Parameter-Efficient Fine-Tuning (QLoRA)

**Author / Student:** College Capstone Project  
**Target Model:** `Qwen/Qwen2.5-1.5B-Instruct` (Apache 2.0 License)  
**Hardware:** Google Colab Free Tier (Tesla T4 GPU, 15GB VRAM)  
**Technique:** 4-bit NormalFloat (NF4) Quantization + LoRA (Low-Rank Adaptation)

---

### 📌 Project Workflow Overview
1. **Hardware & Environment Check:** Verify CUDA, Tesla T4 GPU, and VRAM.
2. **Install Cloud Dependencies:** Install `transformers`, `peft`, `bitsandbytes`, `trl`, `datasets`, and `gradio`.
3. **Mount Google Drive:** Persist trained LoRA adapter weights across ephemeral Colab sessions.
4. **Dataset Setup:** Load and inspect the verified `interview_train.jsonl` and `interview_eval.jsonl` datasets.
5. **Instruction Formatting:** Convert records into standard ChatML conversation format.
6. **QLoRA Setup:** Quantize base model to 4-bit NormalFloat and configure LoRA adapters on attention/MLP projections.
7. **Fine-Tuning:** Train adapter using Hugging Face `SFTTrainer` with conservative memory settings.
8. **Save Adapter:** Export trained adapter weights (`adapter_config.json`, `adapter_model.safetensors`).
9. **Benchmarking:** Compare Base Model vs Fine-Tuned Model on held-out evaluation examples.
10. **Cloud Gradio Launch:** Serve the full interactive web application with a temporary public share link (`.gradio.live`) for live demonstration!


In [24]:
import os
import zipfile

archive = "/content/InterviewMate-colab.zip"
project_dir = "/content/InterviewMate"

with zipfile.ZipFile(archive, "r") as z:
    z.extractall(project_dir)

os.chdir(project_dir)

print("Project directory:", os.getcwd())
print("Files:", os.listdir("."))

Project directory: /content/InterviewMate
Files: ['README.md', '=0.43.1', '.gradio', 'src', '=0.8.6', 'requirements.txt', '=0.11.0', 'scripts', 'colab', 'tests', '=0.30.0', '=4.31.0', 'results', 'app.py', 'saved_adapter', '=2.19.0', '.gitignore', '=4.41.0', 'data']


## Step 1: Verify Hardware & GPU Environment
Before running, make sure your Colab runtime is set to GPU:
👉 Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.


In [3]:
import os
import sys
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ GPU Detected: {device_name}")
    print(f"✅ Total VRAM: {total_mem:.2f} GB")
    if total_mem < 12.0:
        print("⚠️ Warning: Less than 12GB VRAM detected. Will configure conservative batch sizes.")
else:
    print("❌ ERROR: No GPU detected! Please enable GPU via Runtime > Change runtime type > T4 GPU.")
    print("Inference on CPU is possible for demo, but fine-tuning requires a GPU.")

# Display nvidia-smi
!nvidia-smi


Python Version: 3.13.16
PyTorch Version: 2.11.0+cu130
CUDA Available: True
✅ GPU Detected: Tesla T4
✅ Total VRAM: 14.56 GB
Wed Oct 28 08:37:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                

## Step 2: Install Cloud Dependencies
We install modern Hugging Face libraries compatible with CUDA 12 and 4-bit BitsAndBytes quantization.


In [4]:
# Install latest compatible training and inference libraries
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers>=4.41.0 peft>=0.11.0 accelerate>=0.30.0 bitsandbytes>=0.43.1 datasets>=2.19.0 trl>=0.8.6 gradio>=4.31.0 scipy sentencepiece protobuf pydantic

print("✅ Dependencies successfully installed!")


✅ Dependencies successfully installed!


## Step 3: Mount Google Drive (Recommended for Persistence)
Google Colab sessions are ephemeral (files are deleted when the session disconnects). Mounting Google Drive lets us save the trained adapter permanently.


In [5]:
from google.colab import drive
import os

MOUNT_DRIVE = True  # Set to False if you do not wish to mount Drive

if MOUNT_DRIVE:
    try:
        drive.mount('/content/drive')
        DRIVE_SAVE_DIR = '/content/drive/MyDrive/InterviewMate_Adapter'
        os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
        print(f"✅ Google Drive mounted successfully. Adapter will be saved to: {DRIVE_SAVE_DIR}")
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}. Will save adapter locally to './saved_adapter'.")
        DRIVE_SAVE_DIR = './saved_adapter'
else:
    DRIVE_SAVE_DIR = './saved_adapter'
    print("Drive mounting skipped. Adapter will be saved locally.")


Mounted at /content/drive
✅ Google Drive mounted successfully. Adapter will be saved to: /content/drive/MyDrive/InterviewMate_Adapter


## Step 4: Load and Validate the InterviewMate Dataset
If you cloned the project repository into Colab, the files will be in `data/`.  
Otherwise, the code below automatically creates or downloads the verified training and evaluation datasets.


In [6]:
import json
import os

# Check if data directory exists; if not, clone or generate
if not os.path.exists("data/interview_train.jsonl"):
    print("Data files not found in working directory. Cloning repository or generating data...")
    # Option A: Clone from GitHub if hosted:
    # !git clone https://github.com/YOUR_USERNAME/interviewmate.git .

    # Option B: Auto-generate using the dataset script
    if os.path.exists("scripts/generate_dataset.py"):
        !python scripts/generate_dataset.py
    else:
        print("Please ensure interview_train.jsonl and interview_eval.jsonl are uploaded into data/ folder.")

train_path = "data/interview_train.jsonl"
eval_path = "data/interview_eval.jsonl"

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_data = load_jsonl(train_path)
eval_data = load_jsonl(eval_path)

print(f"✅ Loaded {len(train_data)} training records from {train_path}")
print(f"✅ Loaded {len(eval_data)} evaluation records from {eval_path}")

# Display a sample record
sample = train_data[0]
print("\n--- Sample Training Record ---")
print(f"Category: {sample['category']} | Role: {sample['role']}")
print(f"Question: {sample['interview_question']}")
print(f"Candidate Answer ({sample['answer_quality']}): {sample['candidate_answer']}")
print(f"Score: {sample['overall_score']} / 10")


✅ Loaded 123 training records from data/interview_train.jsonl
✅ Loaded 33 evaluation records from data/interview_eval.jsonl

--- Sample Training Record ---
Category: software_development | Role: Backend Engineer
Question: How do database indexes improve query performance, and what are the trade-offs of using them?
Candidate Answer (weak): Indexes make queries fast because they sort data in a table. You should add indexes to every column so that the entire database runs faster.
Score: 4 / 10


## Step 5: Format Dataset for Instruction Fine-Tuning (ChatML Format)
We format each record into standard multi-turn conversation format with `messages`:
- **System:** Evaluator persona and strict JSON schema instructions.
- **User:** The target role, experience level, interview question, and candidate answer.
- **Assistant:** The structured JSON evaluation containing score, rubric breakdown, strengths, weaknesses, actionable advice, improved answer, and follow-up question.


In [7]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are InterviewMate, an expert AI mock interviewer and senior hiring committee member. "
    "Your mission is to evaluate the candidate's interview response with technical precision, "
    "fairness, and constructive coaching. "
    "You MUST respond ONLY with a single valid JSON object containing no markdown outside the JSON. "
    "Follow the requested schema strictly."
)

def format_record_to_chatml(record):
    user_prompt = f"""Candidate Target Role: {record['role']}
Experience Level: {record['experience_level']}
Interview Track: {record['category']}

Interview Question:
"{record['interview_question']}"

Candidate's Answer:
"{record['candidate_answer']}"

Scoring Rubric (Evaluate each metric from 1 to 10):
1. relevance (1-10)
2. accuracy_depth (1-10)
3. clarity_structure (1-10)
4. completeness (1-10)

Provide your evaluation strictly as a valid JSON object matching this schema:
{{
  "overall_score": <number 1-10>,
  "scoring_rubric": {{
    "relevance": <1-10>,
    "accuracy_depth": <1-10>,
    "clarity_structure": <1-10>,
    "completeness": <1-10>
  }},
  "strengths": ["<strength 1>", ...],
  "weaknesses": ["<weakness 1>", ...],
  "actionable_feedback": ["<actionable advice 1>", ...],
  "improved_sample_answer": "<ideal answer>",
  "follow_up_question": "<contextual follow-up>"
}}"""

    assistant_response = json.dumps({
        "overall_score": record["overall_score"],
        "scoring_rubric": record["scoring_rubric"],
        "strengths": record["strengths"],
        "weaknesses": record["weaknesses"],
        "actionable_feedback": record["actionable_feedback"],
        "improved_sample_answer": record["improved_sample_answer"],
        "follow_up_question": record["follow_up_question"]
    }, indent=2, ensure_ascii=False)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
            {"role": "assistant", "content": assistant_response}
        ]
    }

formatted_train = [format_record_to_chatml(r) for r in train_data]
formatted_eval = [format_record_to_chatml(r) for r in eval_data]

hf_train_dataset = Dataset.from_list(formatted_train)
hf_eval_dataset = Dataset.from_list(formatted_eval)

print(f"✅ Formatted {len(hf_train_dataset)} train samples and {len(hf_eval_dataset)} eval samples.")


✅ Formatted 123 train samples and 33 eval samples.


## Step 6: Load Base Model with 4-bit QLoRA Quantization
We load `Qwen/Qwen2.5-1.5B-Instruct` using `BitsAndBytesConfig` in 4-bit NormalFloat (NF4).
- **Base Model Size:** ~3.1 GB in 16-bit FP.
- **Quantized to 4-bit:** ~1.2 GB VRAM!
- **LoRA Rank (r=16, alpha=32):** Adds trainable adapters to attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and feedforward layers (`gate_proj`, `up_proj`, `down_proj`).


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 2. 4-bit NormalFloat Quantization Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load Base Causal LM
print(f"Loading {MODEL_ID} in 4-bit precision...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# 4. Prepare model for LoRA training
model = prepare_model_for_kbit_training(model)

# 5. Define LoRA Adapter Config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Step 7: Fine-Tune using SFTTrainer (Supervised Fine-Tuning)
We train for **3 epochs** using conservative memory hyperparameters:
- `per_device_train_batch_size = 2`
- `gradient_accumulation_steps = 4` (effective batch size = 8)
- `learning_rate = 2e-4` with cosine decay
- `fp16 = True` for NVIDIA T4 Tensor Cores
- Seed = 42 for strict reproducibility


In [16]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "./results"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    optim="paged_adamw_8bit",
    seed=42,
    report_to="none",
    max_length=1200
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_train_dataset,
    eval_dataset=hf_eval_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("🚀 Starting Fine-Tuning with QLoRA...")
train_result = trainer.train()

print("\n✅ Training Completed Successfully!")
print(f"Final Train Loss: {train_result.training_loss:.4f}")

Tokenizing train dataset:   0%|          | 0/123 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/123 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/123 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/123 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/33 [00:00<?, ? examples/s]

...

Truncating eval dataset:   0%|          | 0/33 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/33 [00:00<?, ? examples/s]

🚀 Starting Fine-Tuning with QLoRA...


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.199972,0.911417,0.922824,0.809593,80379.000000
2,0.546337,0.762563,0.675108,0.845931,160758.000000
3,0.550902,0.743980,0.686861,0.848062,241137.000000



✅ Training Completed Successfully!
Final Train Loss: 0.9291


## Step 8: Save Trained LoRA Adapter & Tokenizer
We export the trained adapter parameters to `./saved_adapter` and copy them to Google Drive for permanent backup.


In [17]:
import shutil

LOCAL_ADAPTER_DIR = "./saved_adapter"

print(f"Saving trained LoRA adapter to {LOCAL_ADAPTER_DIR}...")
trainer.model.save_pretrained(LOCAL_ADAPTER_DIR)
tokenizer.save_pretrained(LOCAL_ADAPTER_DIR)

print("Files created in adapter folder:")
for fname in os.listdir(LOCAL_ADAPTER_DIR):
    print(f"  - {fname}")

# Copy to Google Drive if available
if DRIVE_SAVE_DIR != LOCAL_ADAPTER_DIR and os.path.exists("/content/drive"):
    try:
        shutil.copytree(LOCAL_ADAPTER_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
        print(f"✅ Adapter successfully backed up to Google Drive: {DRIVE_SAVE_DIR}")
    except Exception as e:
        print(f"⚠️ Could not copy to Drive: {e}")


Saving trained LoRA adapter to ./saved_adapter...
Files created in adapter folder:
  - README.md
  - tokenizer_config.json
  - tokenizer.json
  - adapter_model.safetensors
  - adapter_config.json
  - chat_template.jinja
✅ Adapter successfully backed up to Google Drive: /content/drive/MyDrive/InterviewMate_Adapter


## Step 9: Benchmark Base Model vs Fine-Tuned Model on Held-out Examples
We test both models on unseen evaluation samples from `data/interview_eval.jsonl` to measure:
1. **JSON Schema Compliance Rate**
2. **Mean Absolute Error (MAE)** against expert ground truth scores
3. **Completeness Rate** (Presence of strengths, weaknesses, improved answer, follow-up)


In [18]:
import time
import json
import re

def test_inference(test_model, test_tok, prompt_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt_text}
    ]
    formatted = test_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = test_tok([formatted], return_tensors="pt").to(test_model.device)

    with torch.no_grad():
        out = test_model.generate(**inputs, max_new_tokens=450, temperature=0.1, do_sample=False)

    in_len = inputs["input_ids"].shape[1]
    return test_tok.decode(out[0][in_len:], skip_special_tokens=True).strip()

# Run a quick sample test
test_sample = eval_data[0]
test_prompt = f"""Target Role: {test_sample['role']}
Track: {test_sample['category']}
Question: "{test_sample['interview_question']}"
Candidate Answer: "{test_sample['candidate_answer']}"

Evaluate according to rubric and provide JSON output."""

print("Evaluating sample using fine-tuned adapter...")
generated_feedback = test_inference(trainer.model, tokenizer, test_prompt)
print("\n--- Generated Model Feedback ---")
print(generated_feedback)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Evaluating sample using fine-tuned adapter...

--- Generated Model Feedback ---
{
  "overall_score": 3,
  "scoring_rationale": {
    "accuracy_depth": 2,
    "contextual_relevance": 1,
    "clarity_structure": 0,
    "completeness": 2
  },
  "strengths": [
    "Identifies that REST uses URLs and GraphQL has custom query languages."
  ],
  "weaknesses": [
    "Lacks structured reasoning or clear communication.",
    "Fails to address trade-offs (scalability, maintenance, error handling), edge cases (cache invalidation), or concrete architectural patterns."
  ],
  "improved_sample_answer": "RESTful APIs follow HTTP conventions like URIs using /path/to/resource?query=param&sort=asc, where resources can be updated via PUT/POST/PATCH and retrieved via GET/DELETE. They expose stateless resource representations through standard HTTP methods. GraphQL provides a typed query language executed against a single endpoint, allowing direct execution of complex queries without external caching mechani

## Step 10: Launch Cloud Gradio Web Application
We launch the interactive Gradio mock interview web interface directly within this Colab notebook!  
Using `share=True`, Gradio generates a temporary public link (`https://xxxx.gradio.live`) that you can open from your Linux Mint laptop or share with your college professors during your presentation!

> **Important Presentation Note:**  
> The public link remains active as long as this Google Colab cell continues executing. When the Colab runtime disconnects, the temporary link will terminate.


In [30]:
# Free up any lingering memory before launching the web server
torch.cuda.empty_cache()

# Launch Gradio interface with public sharing enabled
# (Passes the fine-tuned adapter directory so the app loads the trained model)
!python app.py --model "Qwen/Qwen2.5-1.5B-Instruct" --adapter "./saved_adapter" --share


[App] Initializing model engine...
[ModelLoader] Loading tokenizer for: Qwen/Qwen2.5-1.5B-Instruct
[ModelLoader] Loading base model 'Qwen/Qwen2.5-1.5B-Instruct' (device_map=auto, 4bit=True)...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:01<00:00, 219.82it/s]
[ModelLoader] Found LoRA adapter at './saved_adapter'. Attaching to base model...
[ModelLoader] Successfully attached fine-tuned LoRA adapter!

[InterviewMate] Launching Gradio interface on port 7860 (share=True)...
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://e5b850299d08248084.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848

## 🛠️ Canonical Corrected Gradio Interface

This cell represents the fully debugged, finalized implementation of the **InterviewMate App**.

### 🔧 What this code solves:
1. **Extraction Safeguards**: Extracts only raw JSON substring enclosed between `{` and `}` (omitting markdown fences like ` ```json `).
2. **Fallback Protections**: Recovers gracefully without crashing or returning deceptive zeros when API processing errors occur.
3. **Interface Visual Indicator**: Uses Gradio's native loading animation via `show_progress='full'` so the user is always informed when inference is processing on the backend.

In [32]:
import gradio as gr
import json
import torch
import re

def extract_and_parse_json(text):
    """Extracts and parses a JSON object from raw model outputs that may contain surrounding text or markdown."""
    if not text or not isinstance(text, str):
        raise ValueError("Empty or invalid string received from model.")

    cleaned = text.strip()

    # Locate first { and last } to isolate the JSON block
    match = re.search(r'\{.*\}', cleaned, re.DOTALL)
    if match:
        cleaned = match.group(0)

    return json.loads(cleaned)

def run_mock_interview_app(role, category, experience, question, answer):
    # Formulate prompt cleanly using ChatML instructions structure
    user_prompt = f"""Candidate Target Role: {role}
Experience Level: {experience}
Interview Track: {category}

Interview Question:
\"{question}\"

Candidate's Answer:
\"{answer}\"

Scoring Rubric (Evaluate each metric from 1 to 10):
1. relevance (1-10)
2. accuracy_depth (1-10)
3. clarity_structure (1-10)
4. completeness (1-10)

Provide your evaluation strictly as a valid JSON object matching this schema:
{{
  "overall_score": <number 1-10>,
  "scoring_rubric": {{
    "relevance": <1-10>,
    "accuracy_depth": <1-10>,
    "clarity_structure": <1-10>,
    "completeness": <1-10>
  }},
  "strengths": ["<strength 1>", ...],
  "weaknesses": ["<weakness 1>", ...],
  "actionable_feedback": ["<actionable advice 1>", ...],
  "improved_sample_answer": "<ideal answer>",
  "follow_up_question": "<contextual follow-up>"
}}"""

    try:
        # Execute generation using fine-tuned model and active tokenizer
        raw_output = test_inference(trainer.model, tokenizer, user_prompt)

        # Parse and handle response
        parsed_json = extract_and_parse_json(raw_output)
        formatted_feedback = json.dumps(parsed_json, indent=2, ensure_ascii=False)

        score = parsed_json.get("overall_score", "N/A")
        rubric = parsed_json.get("scoring_rubric", {})
        rubric_str = "\n".join([f"• {k.title()}: {v}/10" for k, v in rubric.items()]) if rubric else ""

        strengths_list = parsed_json.get("strengths", [])
        strengths = "\n".join([f"• {s}" for s in strengths_list]) if isinstance(strengths_list, list) else str(strengths_list)

        weaknesses_list = parsed_json.get("weaknesses", [])
        weaknesses = "\n".join([f"• {w}" for w in weaknesses_list]) if isinstance(weaknesses_list, list) else str(weaknesses_list)

        advice_list = parsed_json.get("actionable_feedback", [])
        advice = "\n".join([f"• {a}" for a in advice_list]) if isinstance(advice_list, list) else str(advice_list)

        improved = parsed_json.get("improved_sample_answer", "")
        follow_up = parsed_json.get("follow_up_question", "")

        display_score = f"{score} / 10" if rubric_str == "" else f"{score} / 10\n\nRubric Breakdown:\n{rubric_str}"
        return display_score, strengths, weaknesses, advice, improved, follow_up, formatted_feedback

    except Exception as e:
        fallback_output = locals().get('raw_output', 'Model returned empty response')
        return "Error during evaluation", f"Details: {str(e)}", "", "", "", "", f"Raw Output:\n{fallback_output}"

# Design custom soft theme Layout
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎓 InterviewMate: Advanced AI Mock Interview Evaluator")
    gr.Markdown("Enter details below. Click **Evaluate Response** to launch active inference. An overlay spinner will indicate processing state.")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📝 Step 1: Input Setup")
            role = gr.Textbox(label="Target Role", value="Backend Engineer")
            category = gr.Dropdown(label="Interview Track", choices=["software_development", "system_design", "behavioral"], value="software_development")
            experience = gr.Dropdown(label="Experience Level", choices=["Junior", "Mid-Level", "Senior"], value="Mid-Level")

            question = gr.Textbox(
                label="Interview Question",
                value="How do database indexes improve query performance, and what are the trade-offs of using them?",
                lines=3
            )
            answer = gr.Textbox(
                label="Candidate's Answer",
                value="Database indexes help the database locate rows without scanning the entire table. They can speed up SELECT queries, but require additional storage and can slow down INSERT, UPDATE, and DELETE operations because the indexes must be maintained.",
                lines=4
            )
            submit_btn = gr.Button("🚀 Evaluate Response", variant="primary")

        with gr.Column(scale=1.5):
            gr.Markdown("### 📊 Step 2: Live AI Evaluation Results")
            with gr.Row():
                overall_score = gr.Textbox(label="Overall Score / Rubric Breakdown", interactive=False, lines=5)

            with gr.Tab("🎯 Performance Summary"):
                strengths = gr.Textbox(label="Strengths", lines=4, interactive=False)
                weaknesses = gr.Textbox(label="Weaknesses", lines=4, interactive=False)
                advice = gr.Textbox(label="Actionable Feedback", lines=4, interactive=False)

            with gr.Tab("💡 Model Suggestions"):
                improved = gr.Textbox(label="Improved Sample Answer", lines=6, interactive=False)
                follow_up = gr.Textbox(label="Contextual Follow-up Question", lines=3, interactive=False)

            with gr.Tab("📄 Raw JSON Output"):
                raw_json = gr.Code(label="Raw JSON API Response", language="json")

    # Run inference with active spinner
    submit_btn.click(
        fn=run_mock_interview_app,
        inputs=[role, category, experience, question, answer],
        outputs=[overall_score, strengths, weaknesses, advice, improved, follow_up, raw_json],
        show_progress="full"
    )

# Launch Web Application
demo.launch(share=True, debug=True)

/tmp/ipykernel_504/3961266036.py:86: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/usr/local/lib/python3.13/dist-packages/gradio/layouts/column.py:59: UserWarning: 'scale' value should be an integer. Using 1.5 will cause issues.
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8248e9676b5748f493.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://8248e9676b5748f493.gradio.live
